# Tool

<div class="alert alert-info">

Namespace `langchain.mcp` yêu cầu `langchain[mcp]>=1.4.0` và hiện đang ở phiên bản beta. API có thể sẽ thay đổi.

</div>

[`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter) là cầu nối giữa MCP server và LangChain agent: nó khám phá các tool mà server công bố và chuyển đổi chúng thành các LangChain tool tiêu chuẩn. Hãy truyền các tool lấy từ `list_tools()` vào [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) giống như bạn làm với bất kỳ LangChain tool nào khác.

Trang này tập trung vào những điểm đặc thù của cầu nối đó: cách nhận diện MCP tool, kiểm soát việc thực thi, xử lý output của chúng, và cách phản hồi khi server cần input trong quá trình gọi. Để xem ví dụ đầy đủ về discovery và agent runnable, hãy xem [MCP quickstart](https://docs.langchain.com/oss/python/langchain/mcp).

## Sử dụng các MCP tool trong agent

Khám phá danh mục tool của server bằng [`MCPAdapter.list_tools`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter/list_tools), sau đó đưa các tool trả về cho [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent). Từ góc nhìn của agent, chúng hoạt động giống hệt như các LangChain tool thông thường: model chọn một tool, LangChain gọi thực thi tool đó, và [`ToolMessage`](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage) kết quả sẽ được trả về cho model.

In [ ]:
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter


async def run_agent(server) -> dict:
    # Khám phá các tool của server, sau đó đưa chúng cho agent giống như
    # bất kỳ LangChain tool nào khác. Các tool này giữ tham chiếu đến client,
    # nên agent vẫn còn sử dụng được trong suốt vòng đời của adapter context.
    async with MCPAdapter(server) as adapter:
        tools = await adapter.list_tools()
        agent = create_agent("claude-sonnet-5", tools)
        return await agent.ainvoke(
            {
                "messages": [
                    {"role": "user", "content": "Dự báo thời tiết ở Oslo thế nào?"}
                ]
            }
        )

[**Xem trace ví dụ**]("https://smith.langchain.com/public/025418ad-e7bc-43a3-b700-d1a78b3a4856/r"): Mở một LangSmith run công khai cho ví dụ này.

Để biết hướng dẫn tổng quát về cách định nghĩa, liên kết, và sử dụng LangChain tool, hãy xem [Tools](https://docs.langchain.com/oss/python/langchain/tools). Để tìm hiểu về nhiều MCP server cùng danh mục tool được đặt namespace riêng của chúng, hãy xem [Connections](https://docs.langchain.com/oss/python/langchain/mcp/connections#multiple-servers).

## Xử lý output của tool

Kết quả trả về từ MCP tool được chuyển thành các giá trị thuần LangChain: nội dung mà model có thể đọc được, một artifact chứa dữ liệu có cấu trúc, và trạng thái [`ToolMessage`](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage) giúp phân biệt lỗi do server báo về với lỗi do sự cố truyền tải.

### Nội dung đa phương thức

Kết quả trả về từ một MCP tool sẽ đến dưới dạng các [content block](https://docs.langchain.com/oss/python/langchain/messages#standard-content-blocks) của LangChain. Nội dung hình ảnh và file sẽ được chuyển đổi thành các block `image` và `file` chuẩn hóa, đi kèm với `text`, vì vậy một tool trả về ảnh chụp màn hình sẽ đến với model dưới dạng một image block:

In [ ]:
from langchain.mcp import MCPAdapter


async def access_multimodal_tool_content(server) -> None:
    async with MCPAdapter(server) as adapter:
        [screenshot] = await adapter.list_tools()

    # Kết quả từ MCP đến dưới dạng các content block của LangChain. Nội dung
    # hình ảnh và file được chuyển đổi thành các block `image`/`file` chuẩn hóa,
    # đi kèm với `text`.
    message = await screenshot.ainvoke(
        {"name": "take_screenshot", "args": {}, "id": "1", "type": "tool_call"}
    )
    for block in message.content_blocks:
        if block["type"] == "text":
            print(f"Văn bản: {block['text']}")
        elif block["type"] == "image":
            print(f"Kiểu MIME của ảnh: {block.get('mime_type')}")
            print(
                f"Ảnh base64: {block.get('base64', '')[:20]}..."
            )

### Nội dung có cấu trúc

Khi một tool trả về nội dung có cấu trúc, adapter sẽ đính kèm nó vào [`ToolMessage`](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage) dưới dạng một artifact thay vì gộp nó vào phần text mà model có thể nhìn thấy. Chạy agent, sau đó đọc `artifact` từ các [`ToolMessage`](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage) trong kết quả trả về:

In [ ]:
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter
from langchain.messages import ToolMessage


async def run_agent_structured(server) -> dict:
    async with MCPAdapter(server) as adapter:
        tools = await adapter.list_tools()
        agent = create_agent("claude-sonnet-5", tools)
        result = await agent.ainvoke(
            {"messages": [{"role": "user", "content": "Tra cứu thông tin user 42."}]}
        )

    # Adapter chỉ gán `artifact` khi tool trả về nội dung có cấu trúc, nên
    # một artifact khác None luôn chứa `structured_content`.
    for message in result["messages"]:
        if isinstance(message, ToolMessage) and message.artifact is not None:
            structured = message.artifact["structured_content"]
            print(f"Nội dung có cấu trúc: {structured}")

    return result

Artifact này là một `MCPToolArtifact`, với trường `structured_content` chứa `structuredContent` từ kết quả trả về của tool. Một tool không trả về nội dung có cấu trúc sẽ khiến `artifact` giữ giá trị `None`.

### Lỗi

Kết quả trả về từ một MCP tool mang theo cờ `isError`. Khi server báo `isError=True`, adapter sẽ chuyển đổi nó thành một [`ToolMessage`](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage) với `status="error"` mang theo thông báo gốc từ server, để agent có thể đọc và tự điều chỉnh:

In [ ]:
from langchain.mcp import MCPAdapter


async def divide_by_zero(server):
    async with MCPAdapter(server) as adapter:
        [divide] = await adapter.list_tools()

    # Một lỗi do server báo về (isError=True) sẽ đến với model dưới dạng
    # một ToolMessage thất bại, để agent có thể đọc thông báo gốc từ server
    # và thử lại. Trong khi đó, lỗi truyền tải (transport) hoặc lỗi phiên làm
    # việc (session) vẫn sẽ raise exception, vì model không thể xử lý được
    # trường hợp kết nối bị ngắt.
    return await divide.ainvoke(
        {"name": "divide", "args": {"a": 10, "b": 0}, "id": "1", "type": "tool_call"}
    )

Lỗi do server báo về sẽ đến với model dưới dạng một tool message thất bại, nhưng lỗi truyền tải hoặc lỗi phiên làm việc thì sẽ raise exception, vì model không thể xử lý được một kết nối bị ngắt đột ngột.

## Metadata của tool

Mỗi tool đã được chuyển đổi có thể mang theo nguồn gốc MCP của nó dưới namespace `mcp` trong metadata của LangChain tool:

```python
tool.metadata
# {
#     "mcp": {
#         "tool": {
#             "annotations": {
#                 "destructive_hint": True,
#                 "read_only_hint": False,
#             },
#             "_meta": {"origin": "crm"},
#         },
#         "server": {
#             "name": "crm",
#             "version": "2.1.0",
#         },
#     },
# }
```

Mọi trường lồng nhau đều là tùy chọn: một server có thể cung cấp annotation cho tool, `_meta`, thông tin định danh server, bất kỳ tổ hợp nào trong số đó, hoặc không cung cấp gì cả. `annotations` chứa các hint của MCP như `read_only_hint` và `destructive_hint`; `_meta` là metadata không xác định cấu trúc (opaque) do server cung cấp; còn `server` xác định danh tính của triển khai MCP đã công bố tool đó.

Hãy đọc metadata tùy chọn một cách phòng thủ, để khi thiếu một trường nào đó thì trả về giá trị mặc định thay vì raise exception:

In [ ]:
from langchain.tools import BaseTool


def is_destructive(tool: BaseTool) -> bool:
    """Đọc destructive hint của MCP từ metadata của tool trong adapter."""
    # Nối chuỗi `.get` với giá trị mặc định để một tool bị thiếu bất kỳ
    # trường lồng nhau nào cũng trả về False thay vì raise exception.
    annotations = (
        (tool.metadata or {}).get("mcp", {}).get("tool", {}).get("annotations", {})
    )
    return annotations.get("destructive_hint", False)

## Human-in-the-loop

Việc đọc annotation cho phép bạn kiểm soát một tool dựa trên những gì server khai báo về nó, thay vì phải hardcode tên tool. Một MCP server có thể đánh dấu một tool là "có tính phá hủy" (destructive) thông qua annotation `destructiveHint`, và [`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter) sẽ hiển thị thông tin này tại `metadata["mcp"]["tool"]["annotations"]["destructive_hint"]`.

Hãy cung cấp cho [`InterruptOnConfig`](https://reference.langchain.com/python/langchain/agents/middleware/human_in_the_loop/InterruptOnConfig) một predicate `when`: một callable nhận vào [`ToolCallRequest`](https://reference.langchain.com/python/langgraph.prebuilt/tool_node/ToolCallRequest) đang chờ xử lý và trả về liệu lệnh gọi đó có cần được phê duyệt hay không. Hãy đọc destructive hint từ metadata một lần duy nhất khi nạp tool, sau đó để callable quyết định cho từng lệnh gọi, nhờ đó một cấu hình duy nhất có thể bao quát mọi destructive tool mà một server công bố, mà không cần hardcode tên tool:

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.agents.middleware.human_in_the_loop import InterruptOnConfig
from langchain.tools import BaseTool
from langchain.tools.tool_node import ToolCallRequest


def is_destructive(tool: BaseTool) -> bool:
    """Đọc destructive hint của MCP từ metadata của tool trong adapter."""
    annotations = (
        (tool.metadata or {}).get("mcp", {}).get("tool", {}).get("annotations", {})
    )
    return annotations.get("destructive_hint", False)


async def gate_destructive_tools(server):
    async with MCPAdapter(server) as adapter:
        tools = await adapter.list_tools()

        # Đọc destructive hint từ metadata một lần, sau đó để một callable
        # quyết định cho từng lệnh gọi. Một cấu hình duy nhất bao quát mọi
        # destructive tool mà một server công bố, mà không cần hardcode tên tool.
        destructive = {tool.name for tool in tools if is_destructive(tool)}

        def needs_approval(request: ToolCallRequest) -> bool:
            return request.tool_call["name"] in destructive

        gate = InterruptOnConfig(
            allowed_decisions=["approve", "reject"], when=needs_approval
        )
        interrupt_on: dict[str, bool | InterruptOnConfig] = {
            tool.name: gate for tool in tools
        }
        return create_agent(
            "claude-sonnet-5",
            tools,
            middleware=[HumanInTheLoopMiddleware(interrupt_on=interrupt_on)],
            checkpointer=InMemorySaver(),
        )

Khi agent gọi một tool bị predicate kiểm soát, quá trình chạy sẽ tạm dừng. Phê duyệt để cho phép tool chạy, hoặc từ chối để bỏ qua tool đó và thông báo cho model:

In [ ]:
from langgraph.types import Command

# Phê duyệt lệnh gọi mang tính phá hủy đang chờ xử lý và tiếp tục chạy.
resumed = await agent.ainvoke(Command(resume={"decisions": [{"type": "approve"}]}), config)

Predicate cũng có thể thấy các tham số của lệnh gọi thông qua `request.tool_call["args"]`, nhờ đó một tool có thể chạy tự do với các input an toàn và chỉ tạm dừng đối với những input rủi ro, chẳng hạn như một lệnh gọi `delete_file` nhắm vào một đường dẫn được bảo vệ. Kết hợp cả hai yếu tố để chỉ kiểm soát một tool khi cả loại tool lẫn tham số của nó đều đáng để làm vậy.

Để xem đầy đủ quy trình phê duyệt, hãy xem [Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop).

## Request từ server trong quá trình thực thi tool

Phần lớn các tool hoàn tất mà không cần hỏi lại client bất cứ điều gì trong quá trình gọi. Khi một server thực sự cần input, [`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter) sẽ tự động trả lời [elicitation](https://modelcontextprotocol.io/specification/draft/client/elicitation) thông qua một [`interrupt`](https://reference.langchain.com/python/langgraph/types/interrupt) của LangGraph.

### Elicitation

[Elicitation](https://modelcontextprotocol.io/specification/draft/client/elicitation) là cơ chế của MCP để một server yêu cầu input ngay giữa một lệnh gọi tool. Khi server cần input, request đó sẽ được đưa lên dưới dạng một LangGraph interrupt, để người đang xem xét công việc của agent có thể trả lời và quá trình chạy sẽ tiếp tục:

In [ ]:
from typing import Any

from langchain.agents import create_agent
from langchain.mcp import MCPAdapter
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


async def book_with_elicitation(server) -> dict:
    # Elicitation được xử lý tự động: khi một server cần input giữa lệnh gọi,
    # adapter sẽ đưa câu hỏi đó lên dưới dạng một LangGraph `interrupt()`,
    # để người đang xem xét công việc của agent có thể trả lời và quá trình
    # chạy sẽ tiếp tục.
    async with MCPAdapter(server) as adapter:
        tools = await adapter.list_tools()

        # Việc tiếp tục một quá trình chạy đang tạm dừng cần có cơ chế lưu
        # trạng thái (persistence), để quá trình bị interrupt có nơi để chờ.
        agent = create_agent("claude-sonnet-5", tools, checkpointer=InMemorySaver())
        config: Any = {"configurable": {"thread_id": "booking-1"}}

        paused = await agent.ainvoke(
            {"messages": [{"role": "user", "content": "Đặt một bàn cho 4 người."}]}, config
        )
        [interrupt] = paused["__interrupt__"]
        [question] = interrupt.value["requests"]

        # Các câu trả lời được đánh khóa (key) theo request key riêng của
        # server, nên không cần phải theo dõi (track) bất cứ điều gì qua
        # thời gian tạm dừng. `decline` hoặc `cancel` sẽ từ chối.
        answer = {"action": "accept", "content": {"date": "2026-09-14"}}
        return await agent.ainvoke(
            Command(resume={"responses": {question["key"]: answer}}), config
        )

Một vài điều cần lưu ý:

* **Elicitation được bật mặc định.** Adapter trang bị cho mọi client mà nó tạo ra để công bố khả năng này và điều khiển vòng lặp interrupt. Một client dựng sẵn đã có sẵn trình xử lý elicitation riêng sẽ được tôn trọng thay vì bị ghi đè.
* **Việc tiếp tục cần có cơ chế lưu trạng thái.** Hãy gắn một [checkpointer](https://docs.langchain.com/oss/python/langchain/short-term-memory) để quá trình chạy bị interrupt có nơi để chờ.
* **Câu trả lời được đánh khóa theo request key của server.** Tiếp tục chạy bằng `Command(resume={"responses": {key: answer}})`. `action` của mỗi câu trả lời có thể là `accept` (đi kèm `content` khớp với schema của request), `decline` (câu trả lời bị từ chối, lệnh gọi vẫn tiếp tục), hoặc `cancel` (toàn bộ lệnh gọi bị hủy bỏ).

Kiểu dữ liệu của interrupt payload và answer nằm trong `langchain.mcp.elicitation`.

Chỉ có elicitation mới được trả lời theo cách này. Một server yêu cầu [sampling](https://modelcontextprotocol.io/specification/2025-06-18/client/sampling) (chạy một lượt hoàn thành LLM) hoặc [roots](https://modelcontextprotocol.io/specification/2025-06-18/client/roots) (các đường dẫn cục bộ có thể truy cập) thay vào đó sẽ raise `NotImplementedError`, vì giao thức hiện đại, không lưu phiên (sessionless) không có kênh phản hồi trực tiếp (live back-channel) cho những request đó. Xem [Sampling and roots](https://docs.langchain.com/oss/python/migrate/langchain-mcp-adapters#sampling-and-roots).

<div class="alert alert-info">

Elicitation dựa trên interrupt chỉ trả lời được cho một server trả về request của nó dưới dạng `InputRequiredResult` (vòng lặp input-required của giao thức hiện đại). Một server chỉ đẩy (push) elicitation qua một phiên bắt tay (handshake) kiểu cũ thì không thể được trả lời theo cách này.

</div>

## Xem thêm

* [Content blocks](https://docs.langchain.com/oss/python/langchain/messages#standard-content-blocks)
* [Tools](https://docs.langchain.com/oss/python/langchain/tools)
* [Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
* [FastMCP calling tools](https://gofastmcp.com/clients/tools)
* [FastMCP client elicitation](https://gofastmcp.com/clients/elicitation)
* [FastMCP server elicitation](https://gofastmcp.com/servers/elicitation)
* [MCP elicitation specification](https://modelcontextprotocol.io/specification/draft/client/elicitation)
* [MCP tool annotations](https://modelcontextprotocol.io/specification/draft/server/tools)